# 03 - Evaluación

En este notebook evaluamos en detalle los resultados obtenidos en el modelado y extraemos conclusiones sobre el rendimiento de cada modelo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
preds = pd.read_csv('predicciones.csv')
preds.head()

FileNotFoundError: [Errno 2] No such file or directory: 'predicciones.csv'

In [ ]:
y_test       = preds['y_test']
y_pred_lr    = preds['y_pred_lr']
y_pred_ridge = preds['y_pred_ridge']
y_pred_lasso = preds['y_pred_lasso']

## Métricas de evaluación

Usamos tres métricas para evaluar los modelos:

- **RMSE** (Root Mean Squared Error): mide el error medio en las mismas unidades que `quality`. Penaliza más los errores grandes.
- **MAE** (Mean Absolute Error): error medio absoluto. Nos dice en promedio cuántos puntos se equivoca el modelo en la escala de calidad.
- **R²**: proporción de la varianza de `quality` que explica el modelo. Un valor de 1 sería perfecto, 0 significaría que el modelo no aporta nada.

In [ ]:
modelos = {
    'Regresión Lineal': y_pred_lr,
    'Ridge': y_pred_ridge,
    'Lasso': y_pred_lasso
}

resultados = pd.DataFrame([
    {
        'Modelo': nombre,
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE':  mean_absolute_error(y_test, y_pred),
        'R²':   r2_score(y_test, y_pred)
    }
    for nombre, y_pred in modelos.items()
]).set_index('Modelo').round(4)

resultados

Los tres modelos obtienen resultados prácticamente idénticos. Ridge y Lasso no mejoran de forma significativa a la regresión lineal simple, lo que indica que la multicolinealidad detectada en el EDA no estaba distorsionando los coeficientes de forma importante.

## Sobreajuste

In [ ]:
# estos valores vienen del notebook de modelado
print('Regresión Lineal:')
print(f'  R² train: 0.3700')
print(f'  R² test:  0.3887')

El R² en test (0.3887) es ligeramente superior al de train (0.3700), lo que descarta cualquier sobreajuste. El modelo generaliza bien a datos nuevos.

## Predicciones vs valores reales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, y_pred) in zip(axes, modelos.items()):
    ax.scatter(y_test, y_pred, alpha=0.3, color='steelblue', edgecolors='none')
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            color='tomato', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Valor real')
    ax.set_ylabel('Predicción')
    ax.set_title(nombre)

plt.suptitle('Predicciones vs valores reales', y=1.01)
plt.tight_layout()
plt.show()

## Distribución de residuos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, y_pred) in zip(axes, modelos.items()):
    residuos = y_test - y_pred
    ax.hist(residuos, bins=30, color='steelblue', edgecolor='white')
    ax.axvline(0, color='tomato', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Error (real - predicción)')
    ax.set_title(nombre)

plt.suptitle('Distribución de residuos', y=1.01)
plt.tight_layout()
plt.show()

Los residuos se distribuyen de forma aproximadamente centrada en 0 en los tres modelos, lo que indica que no hay sesgo sistemático.

## Error por clase de quality

In [ ]:
error_por_clase = pd.DataFrame({
    'quality': y_test.values,
    'error_lr': np.abs(y_test.values - y_pred_lr),
    'error_ridge': np.abs(y_test.values - y_pred_ridge),
    'error_lasso': np.abs(y_test.values - y_pred_lasso)
})

error_por_clase.groupby('quality')[['error_lr', 'error_ridge', 'error_lasso']].mean().round(3)

## Conclusiones

Los tres modelos obtienen resultados muy similares: RMSE=0.725, MAE=0.562 y R²=0.389. La regularización (Ridge y Lasso) no aporta mejora significativa respecto a la regresión lineal simple, lo que sugiere que la multicolinealidad entre variables no era suficientemente grave como para afectar al modelo.

Lasso ha reducido a 0 los coeficientes de `density` y `free sulfur dioxide`, confirmando que estas variables aportaban información redundante o poco relevante para predecir la calidad. Las variables con mayor peso en el modelo son `alcohol` (0.295) y `volatile acidity` (-0.183), como podiamos predecir en el EDA.

No hay sobreajuste: el R² en test (0.3887) es incluso ligeramente superior al de train (0.3700), lo que indica que el modelo generaliza bien.

El R² de 0.39 es un resultado coherente. La calidad del vino es una valoración subjetiva y depende de factores sensoriales que no están recogidos en las variables físico-químicas del dataset. El MAE de 0.56 indica que el modelo se equivoca en promedio poco más de medio punto en la escala de calidad (3-8), lo que es razonable.

La principal limitación es el desbalanceo del dataset: al haber muy pocos vinos de calidad 3 y 8, el modelo falla más en los extremos y tiende a predecir valores intermedios.